# 식탁 물체 인식 — YOLO11 학습 (Colab, T4/L4 GPU)

1. 노트북: `python3 ~/cobot2_ws/src/cobot2/assist_arm/tools/yolo_dataset.py export --only_labeled` → `~/data/assist_arm/export/yolo_table_<시각>.zip`
2. zip 을 Drive **내 드라이브**에 올리기
3. Colab [런타임 → 런타임 유형 변경 → GPU (T4 또는 L4)] → 아래 셀 차례로 실행
4. 끝나면 Drive `assist_table_runs/<이름>/weights/best.pt` 를 노트북 `~/corecode/yolo_train/assist_table/best.pt` 로 받기
   → 컨테이너에서 `ros2 run assist_yolo assist_detection --ros-args -p model_path:=/yolo_train/assist_table/best.pt`
   (모델에 있는 클래스 이름이 별칭보다 먼저 쓰여서 aliases 는 그대로 둬도 돼요)

클래스: rice_bowl, side_dish, cup, water_bottle, spoon, chopsticks, tissue, cloth (config/yolo_classes.yaml 순서 그대로)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/yolo_table_YYYYMMDD_HHMMSS.zip'   # ← 올린 zip 이름으로 바꾸기
!pip -q install ultralytics
!rm -rf /content/assist_table && unzip -q "$ZIP" -d /content/
!ls /content/assist_table && cat /content/assist_table/data.yaml

## 학습 (작은 데이터면 yolo11n, 박스 수천 개 이상이면 yolo11s)

In [ ]:
from ultralytics import YOLO
import os
os.chdir('/content/assist_table')
MODEL = 'yolo11n.pt'          # 또는 'yolo11s.pt'
model = YOLO(MODEL)
results = model.train(data='data.yaml', epochs=120, imgsz=640, batch=16, patience=30,
                      project='/content/drive/MyDrive/assist_table_runs', name=MODEL.replace('.pt', ''),
                      degrees=10, flipud=0.0, fliplr=0.5, hsv_v=0.3, mosaic=1.0, exist_ok=True)

## 평가 (test 분할) + 예측 확인

In [ ]:
best = f'/content/drive/MyDrive/assist_table_runs/{MODEL.replace(".pt", "")}/weights/best.pt'
m = YOLO(best)
metrics = m.val(data='data.yaml', split='test')
print('mAP50', metrics.box.map50, 'mAP50-95', metrics.box.map)
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('test/images/*'))[:4]:
    r = m(p)[0]; r.save('/content/pred.jpg'); display(Image('/content/pred.jpg', width=480))
print('받을 파일:', best)

## (선택) 노트북 CPU/Arc GPU 용 OpenVINO 변환
assist_detection 은 지금 .pt 를 그대로 읽어요 (컨테이너 CPU). 더 빠르게 하려면 아래로 OpenVINO 폴더를 만들어 같이 받기.

In [ ]:
m.export(format='openvino', imgsz=640, half=False)   # best_openvino_model/ 폴더 생성 → model_path 에 그 폴더 경로